# Solution F2 · When the forest loses: linear relationship, trend and cost

**Type**: Compare

## Objective

Identify the three situations in which a random forest is not the best choice, comparing it with linear/logistic regression: (1) a **linear** relationship with many features (the linear model wins and costs a hundredth), (2) **extrapolation** of a trend (the forest is flat), (3) **cost** of training, prediction and memory
(the forest is hundreds of times heavier). And the situation in which it wins: nonlinear interactions.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import pickle
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def problems():
    rng = np.random.default_rng(0)
    Xl = rng.normal(size=(3000, 60))
    w = np.r_[rng.normal(size=15), np.zeros(45)]
    yl = rng.binomial(1, 1 / (1 + np.exp(-2.5 * Xl @ w / np.linalg.norm(w))))
    Xx = rng.normal(size=(2000, 10))
    yx = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
    yx = np.where(rng.random(2000) < 0.05, 1 - yx, yx)
    t = np.arange(400)
    y_t = 5 + 0.05 * t + 2 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.5, 400)
    F = np.c_[t, np.sin(2 * np.pi * t / 12), np.cos(2 * np.pi * t / 12)]
    return {"linear": (Xl, yl), "xor": (Xx, yx)}, (F, y_t)


classification, (F_trend, y_trend) = problems()

## Commented solution

In [ ]:
rows = {}
for name, (X, y) in classification.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    models = {"logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), "forest": RandomForestClassifier(200, random_state=0, n_jobs=-1)}
    for model_name, m in models.items():
        t0 = time.time(); m.fit(X_tr, y_tr); t_fit = time.time() - t0
        t0 = time.time(); acc = m.score(X_te, y_te); t_predict = time.time() - t0
        rows[(name, model_name)] = {"acc": acc, "t_fit": t_fit, "t_predict": t_predict, "mb": len(pickle.dumps(m)) / 1e6}
cls_table = pd.DataFrame(rows).T
cls_table.round(3)

In [ ]:
tr = F_trend[:, 0] < 300
trend_rmse = {}
for name, m in {"linear": LinearRegression(), "forest": RandomForestRegressor(200, random_state=0, n_jobs=-1)}.items():
    m.fit(F_trend[tr], y_trend[tr])
    trend_rmse[name] = float(np.sqrt(np.mean((m.predict(F_trend[~tr]) - y_trend[~tr]) ** 2)))
print({k: round(v, 2) for k, v in trend_rmse.items()})

In [ ]:
acc = cls_table["acc"].unstack()
linear_wins_on = sorted([p for p in acc.index if acc.loc[p, "logistic"] >= acc.loc[p, "forest"] - 0.01] + (["trend"] if trend_rmse["linear"] < trend_rmse["forest"] else []))
forest_wins_on = sorted(p for p in acc.index if acc.loc[p, "forest"] > acc.loc[p, "logistic"] + 0.1)
fit_ratio = cls_table.loc[("linear", "forest"), "t_fit"] / cls_table.loc[("linear", "logistic"), "t_fit"]
mb_ratio = cls_table.loc[("linear", "forest"), "mb"] / cls_table.loc[("linear", "logistic"), "mb"]
print("the linear model wins (or ties) on:", linear_wins_on, "| the forest wins on:", forest_wins_on)
print(f"cost on 'linear': training {fit_ratio:.0f} times, memory {mb_ratio:.0f} times")

## Check

In [ ]:
with verify():
    check(cls_table.shape == (4, 4) and list(cls_table.columns) == ["acc", "t_fit", "t_predict", "mb"], "4 rows (2 problems x 2 models) and columns acc, t_fit, t_predict, mb", "one row per (problem, model) pair")
    check(set(trend_rmse) == {"linear", "forest"} and trend_rmse["linear"] < 1 and trend_rmse["forest"] > 2.5, "trend: the linear one is off by less than 1, the forest by more than 2.5", "test on months 300-399")
    check(linear_wins_on == ["linear", "trend"] and forest_wins_on == ["xor"], "the linear model wins (or ties) on 'linear' and 'trend'; the forest beats the logistic by over 0.1 on 'xor'", "compare the accuracies and the RMSEs")
    check(cls_table.loc[("xor", "logistic"), "acc"] < 0.6 and cls_table.loc[("xor", "forest"), "acc"] > 0.9, "on the XOR the logistic is at chance (below 0.6), the forest over 0.9", "the XOR is not linear")
    check(fit_ratio > 20 and mb_ratio > 100, "the forest costs over 20 times in training and over 100 times in memory compared with the logistic", "t_fit and mb")
done()

## What you learned

- **Linear relationship with many features:** the logistic (0.80) beats the forest (0.76) and costs a hundredth of the time and memory. When the true link is a linear combination of many variables, a linear model estimates it with few parameters, while trees have to approximate it with many thresholds.
- **Interactions (XOR):** it is the ground of the forest: 0.93 against 0.52 for the logistic. No linear combination separates the four quadrants; trees find the two thresholds and their interaction without you specifying it.
- **Extrapolation:** on a trend the forest is off by more than seven times as much as the line (3.1 against 0.43): beyond the range seen in training it predicts a constant value. You need a model with structure (linear, ARIMA) or a hybrid (M7 C1).
- **Cost:** the forest takes about 0.7 seconds of training and 5-7 MB (200 trees on 1800 rows: they grow with the data, as in the 244 MB of B2), the logistic 14 milliseconds and 3 KB. To serve the model on a device, with low latency or with many models in parallel, the cost matters as much as the accuracy.
- These are not arguments against the forest (it is the best first attempt on tabular data): they are the **scenarios in which to also try a linear model** and compare with the same validation.

## Common pitfall

Using a forest by default without ever trying a linear baseline. On linear relationships and on trends the forest can lose, and the baseline tells you anyway how much the added complexity is worth.

## Interview question

> *Click on a question to show its answer.*

<details><summary>In which cases would you not use a random forest?</summary>

When you need to extrapolate (trends, nonstationary series), when the relationship is linear with many features (a regularized linear model is better and stable), when you need a small, fast model (devices, latency) or simple explanations, and with very sparse high-dimensional data (text). I always compare it with a linear baseline
and with a boosting, using the same validation.

</details>